In [1]:
#!/usr/bin/env python3

# Phase 2 — Step 1: Tokenize Hindi Corpus → Binary Files
#

In [2]:
LANG = "hindi"

# === UPDATE THESE PATHS to match your Kaggle dataset ===
INPUT_DIR = "/kaggle/input/datasets/srinivasu2005/hindi-tokenizer-output/splits"          # dir containing train.txt, val.txt, test.txt
TOKENIZER_PATH = "/kaggle/input/datasets/srinivasu2005/hindi-tokenizer-output/tokenizer/tokenizer.model"  # SentencePiece model

OUTPUT_DIR = "/kaggle/working/hindi_tokenized"
BOS_ID = 2
EOS_ID = 3
IS_JSONL = False  # Set True if your splits are JSONL format

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])

0

In [4]:
import os
import json
import numpy as np
import sentencepiece as spm
import logging
import time

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s")
logger = logging.getLogger("tokenize")


def tokenize_split(sp, input_path, output_path, bos_id=2, eos_id=3, is_jsonl=False):
    """Tokenize a text split and save as flat uint16 binary."""
    all_ids = []
    num_docs = 0
    
    logger.info(f"Tokenizing {input_path} ...")
    start = time.time()
    
    with open(input_path, "r", encoding="utf-8") as f:
        if is_jsonl:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    text = json.loads(line).get("text", "").strip()
                except json.JSONDecodeError:
                    continue
                if not text:
                    continue
                ids = sp.encode(text, out_type=int)
                all_ids.append(bos_id)
                all_ids.extend(ids)
                all_ids.append(eos_id)
                num_docs += 1
                if num_docs % 500_000 == 0:
                    logger.info(f"  {num_docs:,} docs, {len(all_ids):,} tokens")
        else:
            current_doc = []
            for line in f:
                stripped = line.strip()
                if stripped:
                    current_doc.append(stripped)
                else:
                    if current_doc:
                        text = " ".join(current_doc)
                        ids = sp.encode(text, out_type=int)
                        all_ids.append(bos_id)
                        all_ids.extend(ids)
                        all_ids.append(eos_id)
                        num_docs += 1
                        current_doc = []
                        if num_docs % 500_000 == 0:
                            logger.info(f"  {num_docs:,} docs, {len(all_ids):,} tokens")
            if current_doc:
                text = " ".join(current_doc)
                ids = sp.encode(text, out_type=int)
                all_ids.append(bos_id)
                all_ids.extend(ids)
                all_ids.append(eos_id)
                num_docs += 1
    
    arr = np.array(all_ids, dtype=np.uint16)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    arr.tofile(output_path)
    
    elapsed = time.time() - start
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    logger.info(f"  Done: {num_docs:,} docs, {len(all_ids):,} tokens, "
                f"{size_mb:.1f} MB, {elapsed:.1f}s → {output_path}")
    
    return {"num_docs": num_docs, "num_tokens": len(all_ids), "file_size_mb": round(size_mb, 2)}

In [5]:
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
logger.info(f"Loaded tokenizer: vocab_size={sp.get_piece_size()}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
stats = {}

for split in ["train", "val", "test"]:
    ext = ".jsonl" if IS_JSONL else ".txt"
    input_path = os.path.join(INPUT_DIR, f"{split}{ext}")
    if not os.path.exists(input_path):
        logger.warning(f"Not found: {input_path}, skipping")
        continue
    output_path = os.path.join(OUTPUT_DIR, f"{split}.bin")
    stats[split] = tokenize_split(sp, input_path, output_path, BOS_ID, EOS_ID, IS_JSONL)

# Save stats
with open(os.path.join(OUTPUT_DIR, "tokenization_stats.json"), "w") as f:
    json.dump(stats, f, indent=2)

[2026-08-28 04:57:09,836] INFO — Loaded tokenizer: vocab_size=8000
[2026-08-28 04:57:09,842] INFO — Tokenizing /kaggle/input/datasets/srinivasu2005/hindi-tokenizer-output/splits/train.txt ...
[2026-08-28 04:58:06,225] INFO —   500,000 docs, 51,078,617 tokens
[2026-08-28 04:59:03,748] INFO —   1,000,000 docs, 102,672,273 tokens
[2026-08-28 05:00:02,079] INFO —   1,500,000 docs, 153,771,484 tokens
[2026-08-28 05:01:01,817] INFO —   2,000,000 docs, 205,120,465 tokens
[2026-08-28 05:02:03,247] INFO —   2,500,000 docs, 256,373,752 tokens
[2026-08-28 05:03:01,737] INFO —   3,000,000 docs, 307,515,795 tokens
[2026-08-28 05:04:00,461] INFO —   3,500,000 docs, 358,599,020 tokens
[2026-08-28 05:04:58,891] INFO —   4,000,000 docs, 409,970,293 tokens
[2026-08-28 05:05:58,481] INFO —   4,500,000 docs, 461,129,500 tokens
[2026-08-28 05:06:59,298] INFO —   5,000,000 docs, 512,318,676 tokens
[2026-08-28 05:07:58,115] INFO —   5,500,000 docs, 563,794,668 tokens
[2026-08-28 05:08:56,382] INFO —   6,000,

In [6]:
print("\n" + "=" * 60)
print(f"Hindi Tokenization Complete")
print("=" * 60)
for split, s in stats.items():
    print(f"  {split}: {s['num_docs']:,} docs, {s['num_tokens']:,} tokens, {s['file_size_mb']} MB")
print(f"\nOutput: {OUTPUT_DIR}/")
print("Next: Upload output as a Kaggle dataset, then run 03_train_hindi.py")


Hindi Tokenization Complete
  train: 7,752,732 docs, 794,542,939 tokens, 1515.47 MB
  val: 161,854 docs, 16,521,265 tokens, 31.51 MB
  test: 161,317 docs, 16,557,992 tokens, 31.58 MB

Output: /kaggle/working/hindi_tokenized/
Next: Upload output as a Kaggle dataset, then run 03_train_hindi.py
